# release16_release_gate_bc_official_replay_v1

Purpose: replay the selected release-gate-weighted BC actor from `release16_release_gate_bc_weight_screen_v1` on official qlib daily paths before any PPO or actor retention.

Governance:
- Canonical baseline: `soxlbacktest_policy_reimplementation_v1`.
- Direct comparison: `release16_release_gate_bc_weight_screen_v1` and `baseline_current` BC weighting.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024` walk-forward tests.
- Changed layer: `model_selection` / `bc_loss_weighting` / `official_replay`.
- External reference: qlib-converted SOXLBacktest candidate metrics including `legacy_default` and `wf_pass_ytd18_15_15_02`.

Hypothesis: a BC actor that fixes the release-gate slice should improve official replay path metrics without breaking the MaxDD < 20% guard. If it does not pass the walk-forward path gate, the next mechanism is not PPO length; it is official replay target/reward repair or checkpoint scoring.


In [1]:

from __future__ import annotations

import copy
import json
import math
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").exists() and (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.checkpoint_daily_export import build_checkpoint_candidate_daily_export, metrics_from_official_backtest
from src.soxl_quant.app.state_objective_bc import StateObjectiveBCSampleWeightConfig, StateObjectiveBCTrainingConfig, fit_state_objective_bc_actor_from_frame
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = "release16_release_gate_bc_official_replay_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
WEIGHT_SCREEN = ROOT / "runtime/model_selection/card/release16_release_gate_bc_weight_screen_v1"
CONTRACT_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_ytd_alignment_contract_v1"
TARGET_CONTRACT = CONTRACT_CARD / "distillation_target_contract.csv"
REFERENCE_METRICS = ROOT / "runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1/candidate_metrics.csv"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260615
TARGET_RELATIVE_RETURN = 30.0
MAX_DD_PCT = 20.0
STRICT_MAX_DD_PCT = 15.0
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / "official_daily_paths"
DAILY_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=400, lr=0.001, batch_size=128, seed=SEED)
WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "target_fold": "wf_2020_train",
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})
    return series.fillna(False).astype(bool)


def config_from_row(row: pd.Series) -> StateObjectiveBCSampleWeightConfig:
    fields = {name: row[name] for name in StateObjectiveBCSampleWeightConfig.__dataclass_fields__ if name in row.index}
    return StateObjectiveBCSampleWeightConfig(**{k: float(v) if k != "normalize_mean" else bool(v) for k, v in fields.items()})


def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str], *, fold: str, candidate: str) -> dict[str, Any]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = metrics_from_official_backtest(split_metrics=dict(split_metrics), backtest_result=bt)
    export = build_checkpoint_candidate_daily_export(
        backtest_result=bt,
        price_series=price_series,
        output_dir=DAILY_DIR,
        fold=fold,
        candidate=candidate,
        candidate_kind="bc_initialization",
        split_metrics=dict(split_metrics),
        root=ROOT,
    )
    for source_key, target_key in {
        "test_return": "return",
        "test_max_dd": "max_dd",
        "test_avg_pos": "avg_pos",
        "test_buy_and_hold_return": "buy_and_hold_return",
        "test_relative_return": "relative_return",
    }.items():
        if source_key in export.row:
            out[target_key] = float(export.row[source_key])
    out.update({k: v for k, v in export.row.items() if k not in out})
    return out

print("root", ROOT)
print("card", CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_release_gate_bc_official_replay_v1


In [2]:

weight_selection = load_json(WEIGHT_SCREEN / "selection.json")
selected_config_id = str(weight_selection["selected_config"])
config_grid = pd.read_csv(WEIGHT_SCREEN / "config_grid.csv")
configs = {
    "baseline_current": config_from_row(config_grid[config_grid["config_id"].eq("baseline_current")].iloc[0]),
    selected_config_id: config_from_row(config_grid[config_grid["config_id"].eq(selected_config_id)].iloc[0]),
}
if selected_config_id == "baseline_current":
    configs = {"baseline_current": configs["baseline_current"]}

profile = load_profile(PROFILE_NAME)
targets = pd.read_csv(TARGET_CONTRACT)
targets["datetime"] = pd.to_datetime(targets["datetime"]).dt.normalize()
for col in ("training_label_allowed", "is_rebound_cluster", "is_downtrend_state", "release_gate_state", "drawdown_control_state"):
    if col in targets.columns:
        targets[col] = normalize_bool_series(targets[col])

reference_metrics = pd.read_csv(REFERENCE_METRICS)
reference_rows = reference_metrics[
    reference_metrics["candidate"].isin(["legacy_default", "wf_pass_ytd18_15_15_02"])
    & reference_metrics["window"].isin(WALK_FORWARD_WINDOWS)
    & reference_metrics["split_type"].eq("walk_forward_test")
].copy()
reference_rows["relative_return_pct"] = reference_rows["rulebase_return_pct"] - reference_rows["soxl_buyhold_return_pct"]
reference_rows.to_csv(CARD / "soxlbacktest_reference_metrics.csv", index=False)

pd.DataFrame([{"config_id": k, **asdict(v)} for k, v in configs.items()]).to_csv(CARD / "replay_config_grid.csv", index=False)
print("selected_config", selected_config_id, "configs", list(configs))
display(reference_rows[["candidate", "window", "rulebase_return_pct", "rulebase_max_drawdown_pct", "relative_return_pct", "average_position_pct"]])


selected_config release64_no_objective configs ['baseline_current', 'release64_no_objective']


,candidate,window,rulebase_return_pct,rulebase_max_drawdown_pct,relative_return_pct,average_position_pct
0,legacy_default,wf_2020,220.777205,54.275144,161.001897,79.102332
1,legacy_default,wf_2022,-79.396371,84.918053,7.191701,86.692988
2,legacy_default,wf_2023_2024,2.935118,32.461810,5.538546,31.043492
12,wf_pass_ytd18_15_15_02,wf_2020,29.156603,12.785207,-30.618705,17.101107
13,wf_pass_ytd18_15_15_02,wf_2022,-16.044551,18.514585,70.543521,10.231793
14,wf_pass_ytd18_15_15_02,wf_2023_2024,4.505779,1.061804,7.109208,1.322063


In [3]:

init_qlib(QLIB_DATA_PATH)
replay_rows: list[dict[str, Any]] = []

for fold, spec in WALK_FORWARD_WINDOWS.items():
    print("fold", fold)
    dataset = build_dataset(
        train_seg=spec["train"],
        valid_seg=spec["valid"],
        test_seg=spec["test"],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    train_df = segment_frame(dataset, "train")
    test_df = segment_frame(dataset, "test")
    price_frame = dataset.handler.price_frame.droplevel("instrument")["price"].sort_index()
    test_price_series = price_frame.loc[spec["test"][0] : spec["test"][1]]

    train_start = pd.Timestamp(spec["train"][0])
    train_end = pd.Timestamp(spec["train"][1])
    fold_targets = targets[targets["fold"].astype(str).eq(spec["target_fold"])].copy()
    fold_targets = fold_targets[fold_targets["datetime"].between(train_start, train_end)].copy()
    fold_targets = fold_targets[fold_targets["training_label_allowed"].fillna(False).astype(bool)].copy()

    for config_id, weight_config in configs.items():
        print("  config", config_id)
        set_seed(SEED)
        fit = fit_state_objective_bc_actor_from_frame(
            train_df,
            targets=fold_targets,
            profile=profile,
            train_config=TRAIN_CONFIG,
            sample_weight_config=weight_config,
            sample_size=0,
            rng_seed=SEED,
        )
        metrics = evaluate_actor_path(
            fit.actor,
            test_df,
            test_price_series,
            profile,
            spec["test"],
            fold=fold,
            candidate=config_id,
        )
        row = {
            "candidate": config_id,
            "fold": fold,
            "target_rows": int(len(fold_targets)),
            "bc_samples": int(fit.metrics["samples"]),
            "bc_after_accuracy": float(fit.metrics["after_accuracy"]),
            "bc_after_loss": float(fit.metrics["after_loss"]),
            "return_pct": float(metrics.get("return", float("nan"))),
            "max_dd_pct": float(metrics.get("max_dd", float("nan"))),
            "avg_pos": float(metrics.get("avg_pos", float("nan"))),
            "buy_and_hold_return_pct": float(metrics.get("buy_and_hold_return", float("nan"))),
            "relative_return_pp": float(metrics.get("relative_return", float("nan"))),
            "daily_path": metrics.get("daily_path"),
            "daily_path_rows": metrics.get("daily_path_rows"),
            "metric_basis": metrics.get("metric_basis"),
            "raw_backtest_return": metrics.get("raw_backtest_return"),
            "raw_backtest_max_dd": metrics.get("raw_backtest_max_dd"),
            "metric_return_abs_diff": metrics.get("metric_return_abs_diff"),
            "metric_max_dd_abs_diff": metrics.get("metric_max_dd_abs_diff"),
        }
        row["wf_gate_20dd"] = bool(row["relative_return_pp"] >= TARGET_RELATIVE_RETURN and row["max_dd_pct"] <= MAX_DD_PCT)
        row["strict_gate_15dd"] = bool(row["relative_return_pp"] >= TARGET_RELATIVE_RETURN and row["max_dd_pct"] <= STRICT_MAX_DD_PCT)
        replay_rows.append(row)

replay_metrics = pd.DataFrame(replay_rows)
replay_metrics.to_csv(CARD / "bc_official_replay_metrics.csv", index=False)
display(replay_metrics)


[37457:MainThread](2026-06-15 03:54:48,441) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[37457:MainThread](2026-06-15 03:54:48,839) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[37457:MainThread](2026-06-15 03:54:48,839) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


fold wf_2020


[37457:MainThread](2026-06-15 03:54:49,771) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

[37457:MainThread](2026-06-15 03:55:14,446) WARNING - qlib.online operator - [exchange.py:226] - factor.day.bin file not exists or factor contains `nan`. Order using adjusted_price.


[37457:MainThread](2026-06-15 03:55:14,446) WARNING - qlib.online operator - [exchange.py:228] - trade unit 1 is not supported in adjusted_price mode.


backtest loop:   0%|          | 0/251 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,candidate,fold,target_rows,bc_samples,bc_after_accuracy,bc_after_loss,return_pct,max_dd_pct,avg_pos,buy_and_hold_return_pct,relative_return_pp,daily_path,daily_path_rows,metric_basis,raw_backtest_return,raw_backtest_max_dd,metric_return_abs_diff,metric_max_dd_abs_diff,wf_gate_20dd,strict_gate_15dd
0,baseline_current,wf_2020,2218,1904,0.723214,1.184816,73.995545,24.834087,0.396934,58.093571,15.901974,runtime/model_selection/card/release16_release...,252,official_daily_path_rebased_eval_window,73.995545,24.834087,0.0,3.552714e-15,False,False
1,release64_no_objective,wf_2020,2218,1904,0.653887,1.246048,80.375210,21.106156,0.425202,58.093571,22.281639,runtime/model_selection/card/release16_release...,252,official_daily_path_rebased_eval_window,80.375210,21.106156,0.0,7.105427e-15,False,False
2,baseline_current,wf_2022,253,252,0.841270,1.061176,-16.028050,20.401262,0.093902,-86.588072,70.560022,runtime/model_selection/card/release16_release...,250,official_daily_path_rebased_eval_window,-16.028050,20.401262,0.0,3.552714e-15,False,False
3,release64_no_objective,wf_2022,253,252,0.726190,1.174679,-23.837432,26.860565,0.144199,-86.588072,62.750639,runtime/model_selection/card/release16_release...,250,official_daily_path_rebased_eval_window,-23.837432,26.860565,0.0,0.000000e+00,False,False
4,baseline_current,wf_2023_2024,756,755,0.772185,1.134638,6.829392,13.622379,0.224851,-0.356635,7.186027,runtime/model_selection/card/release16_release...,251,official_daily_path_rebased_eval_window,6.829392,13.622379,0.0,0.000000e+00,False,False
5,release64_no_objective,wf_2023_2024,756,755,0.715232,1.187326,13.945428,13.959434,0.197766,-0.356635,14.302062,runtime/model_selection/card/release16_release...,251,official_daily_path_rebased_eval_window,13.945428,13.959434,0.0,3.552714e-15,False,False


In [4]:

comparison_rows: list[dict[str, Any]] = []
for fold in WALK_FORWARD_WINDOWS:
    rows = replay_metrics[replay_metrics["fold"].eq(fold)].set_index("candidate")
    if "baseline_current" in rows.index and selected_config_id in rows.index and selected_config_id != "baseline_current":
        base = rows.loc["baseline_current"]
        sel = rows.loc[selected_config_id]
        comparison_rows.append({
            "fold": fold,
            "selected_config": selected_config_id,
            "selected_return_pct": float(sel["return_pct"]),
            "baseline_return_pct": float(base["return_pct"]),
            "return_delta_pct": float(sel["return_pct"] - base["return_pct"]),
            "selected_relative_return_pp": float(sel["relative_return_pp"]),
            "baseline_relative_return_pp": float(base["relative_return_pp"]),
            "relative_delta_pp": float(sel["relative_return_pp"] - base["relative_return_pp"]),
            "selected_max_dd_pct": float(sel["max_dd_pct"]),
            "baseline_max_dd_pct": float(base["max_dd_pct"]),
            "max_dd_delta_pct": float(sel["max_dd_pct"] - base["max_dd_pct"]),
            "selected_wf_gate_20dd": bool(sel["wf_gate_20dd"]),
            "selected_strict_gate_15dd": bool(sel["strict_gate_15dd"]),
        })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(CARD / "selected_vs_baseline_replay_delta.csv", index=False)

selected_rows = replay_metrics[replay_metrics["candidate"].eq(selected_config_id)].copy()
all_folds_replayed = set(selected_rows["fold"].astype(str)) == set(WALK_FORWARD_WINDOWS)
selected_gate_count = int(selected_rows["wf_gate_20dd"].fillna(False).astype(bool).sum())
selected_strict_gate_count = int(selected_rows["strict_gate_15dd"].fillna(False).astype(bool).sum())
all_selected_gate = bool(all_folds_replayed and selected_gate_count == len(WALK_FORWARD_WINDOWS))
all_selected_strict_gate = bool(all_folds_replayed and selected_strict_gate_count == len(WALK_FORWARD_WINDOWS))

if all_selected_gate:
    decision = "release_gate_weighted_bc_official_replay_passed_no_promotion"
    next_mechanism = "run_fixed_split_ytd_dashboard_replay_and_actor_retention_gate_before_ppo"
else:
    decision = "release_gate_weighted_bc_official_replay_failed_no_promotion"
    next_mechanism = "repair_official_replay_objective_or_checkpoint_scoring_before_ppo"

retention_summary = load_json(RETENTION_GATE).get("summary", {})
selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "selected_config": selected_config_id,
    "all_selected_gate_20dd": all_selected_gate,
    "all_selected_gate_15dd": all_selected_strict_gate,
    "selected_gate_count_20dd": selected_gate_count,
    "selected_gate_count_15dd": selected_strict_gate_count,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
}
write_json(CARD / "selection.json", selection)

summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose_official_bc_replay",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_release_gate_bc_weight_screen_v1",
    "target_window": list(WALK_FORWARD_WINDOWS),
    "changed_layer": ["model_selection", "bc_loss_weighting", "official_replay"],
    "external_reference": "soxlbacktest_qlib_walkforward_baseline_v1 candidate metrics",
    "hypothesis": "A release-gate-weighted BC actor should improve official daily replay if the slice repair transfers to path behavior.",
    "acceptance_criteria": [
        "selected and baseline BC configs are replayed on all walk-forward folds",
        "selected config passes relative_return >= 30pp and MaxDD <= 20% on every fold",
        "strict <=15% MaxDD gate is recorded as additional risk context",
        "no actor retained or promoted from this diagnostic replay",
    ],
    "expected_tradeoff": "Better release-state participation may improve wf_2023_2024 but can raise drawdown or hurt stress folds.",
    "failure_mode": "Slice-fit improvement may not transfer to official qlib path metrics, requiring checkpoint/reward objective repair instead of longer PPO.",
    "comparison_table_row_key": EXPERIMENT,
    "closest_historical_success_case": "release16_release_gate_bc_weight_screen_v1 fixed the wf_2023_2024 release_gate_state slice to 0.818 accuracy.",
    "closest_historical_failure_case": "release16_checkpoint_distillation_bc_readiness_v1 failed wf_2023_2024 release_gate_state with single-action prediction collapse.",
    "why_not_duplicate": "This notebook runs official daily path replay for the selected BC weighting; the prior screen only measured supervised slice fit.",
    "selected_config": selected_config_id,
    "gate_summary": {
        "all_folds_replayed": all_folds_replayed,
        "selected_gate_count_20dd": selected_gate_count,
        "selected_gate_count_15dd": selected_strict_gate_count,
        "required_fold_count": len(WALK_FORWARD_WINDOWS),
        "all_selected_gate_20dd": all_selected_gate,
        "all_selected_gate_15dd": all_selected_strict_gate,
    },
    "retention_gate_current": {
        "passing_profile_count": retention_summary.get("passing_profile_count"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
    },
    "selected_replay_metrics": selected_rows.to_dict("records"),
    "selected_vs_baseline_delta": comparison.to_dict("records"),
    "output_paths": {
        "bc_official_replay_metrics": str((CARD / "bc_official_replay_metrics.csv").relative_to(ROOT)),
        "selected_vs_baseline_replay_delta": str((CARD / "selected_vs_baseline_replay_delta.csv").relative_to(ROOT)),
        "soxlbacktest_reference_metrics": str((CARD / "soxlbacktest_reference_metrics.csv").relative_to(ROOT)),
        "official_daily_paths": str(DAILY_DIR.relative_to(ROOT)),
        "selection": str((CARD / "selection.json").relative_to(ROOT)),
        "summary": str((CARD / "summary.json").relative_to(ROOT)),
    },
    "next_mechanism": next_mechanism,
    "conclusion": (
        "Selected BC weighting passed all official walk-forward daily path gates; still no promotion until fixed split, YTD/dashboard replay, and actor retention gates pass."
        if all_selected_gate
        else "Selected BC weighting fixes supervised release-gate fit but does not pass official walk-forward path gates; do not run PPO until replay objective/checkpoint scoring is repaired."
    ),
}
write_json(CARD / "summary.json", summary)

print("decision", decision)
display(comparison)
display(selected_rows)


decision release_gate_weighted_bc_official_replay_failed_no_promotion


,fold,selected_config,selected_return_pct,baseline_return_pct,return_delta_pct,selected_relative_return_pp,baseline_relative_return_pp,relative_delta_pp,selected_max_dd_pct,baseline_max_dd_pct,max_dd_delta_pct,selected_wf_gate_20dd,selected_strict_gate_15dd
0,wf_2020,release64_no_objective,80.375210,73.995545,6.379665,22.281639,15.901974,6.379665,21.106156,24.834087,-3.727931,False,False
1,wf_2022,release64_no_objective,-23.837432,-16.028050,-7.809382,62.750639,70.560022,-7.809382,26.860565,20.401262,6.459303,False,False
2,wf_2023_2024,release64_no_objective,13.945428,6.829392,7.116036,14.302062,7.186027,7.116036,13.959434,13.622379,0.337055,False,False


,candidate,fold,target_rows,bc_samples,bc_after_accuracy,bc_after_loss,return_pct,max_dd_pct,avg_pos,buy_and_hold_return_pct,relative_return_pp,daily_path,daily_path_rows,metric_basis,raw_backtest_return,raw_backtest_max_dd,metric_return_abs_diff,metric_max_dd_abs_diff,wf_gate_20dd,strict_gate_15dd
1,release64_no_objective,wf_2020,2218,1904,0.653887,1.246048,80.375210,21.106156,0.425202,58.093571,22.281639,runtime/model_selection/card/release16_release...,252,official_daily_path_rebased_eval_window,80.375210,21.106156,0.0,7.105427e-15,False,False
3,release64_no_objective,wf_2022,253,252,0.726190,1.174679,-23.837432,26.860565,0.144199,-86.588072,62.750639,runtime/model_selection/card/release16_release...,250,official_daily_path_rebased_eval_window,-23.837432,26.860565,0.0,0.000000e+00,False,False
5,release64_no_objective,wf_2023_2024,756,755,0.715232,1.187326,13.945428,13.959434,0.197766,-0.356635,14.302062,runtime/model_selection/card/release16_release...,251,official_daily_path_rebased_eval_window,13.945428,13.959434,0.0,3.552714e-15,False,False


## Conclusion

The final cell renders `summary.json`. A replay pass here is not promotion; it only determines whether the selected BC weighting is strong enough to justify fixed/YTD/dashboard replay work.


In [5]:

summary = load_json(CARD / "summary.json")
lines = [
    f"Decision: `{summary['decision']}`.",
    f"Selected config: `{summary['selected_config']}`.",
    f"WF gate count (<=20% MaxDD): {summary['gate_summary']['selected_gate_count_20dd']} / {summary['gate_summary']['required_fold_count']}.",
    f"Strict gate count (<=15% MaxDD): {summary['gate_summary']['selected_gate_count_15dd']} / {summary['gate_summary']['required_fold_count']}.",
    f"Next mechanism: `{summary['next_mechanism']}`.",
    summary["conclusion"],
]
display(Markdown("\n".join(f"- {line}" for line in lines)))


- Decision: `release_gate_weighted_bc_official_replay_failed_no_promotion`.
- Selected config: `release64_no_objective`.
- WF gate count (<=20% MaxDD): 0 / 3.
- Strict gate count (<=15% MaxDD): 0 / 3.
- Next mechanism: `repair_official_replay_objective_or_checkpoint_scoring_before_ppo`.
- Selected BC weighting fixes supervised release-gate fit but does not pass official walk-forward path gates; do not run PPO until replay objective/checkpoint scoring is repaired.